In [2]:
import regex

from collections import defaultdict


text = """
                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.
                     2
  When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep sunken eyes,
  Were an all-eating shame, and thriftless praise.
  How much more praise deserved thy beauty's use,
  If thou couldst answer 'This fair child of mine
  Shall sum my count, and make my old excuse'
  Proving his beauty by succession thine.
    This were to be new made when thou art old,
    And see thy blood warm when thou feel'st it cold.
"""

SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""


def get_chunks(text):
    """Split text into pre-token chunks, then convert each to a tuple of byte values."""
    chunks = regex.findall(SPLIT_PATTERN, text)
    return [tuple(chunk.encode("utf-8")) for chunk in chunks]


def get_pair_counts(chunks):
    counts = defaultdict(int)
    for chunk in chunks:
        for pair in zip(chunk, chunk[1:]):
            counts[pair] += 1
    return counts


def merge_chunks(chunks, pair, new_id):
    a, b = pair
    new_chunks = []
    for chunk in chunks:
        new_chunk = []
        i = 0
        while i < len(chunk):
            if i < len(chunk) - 1 and chunk[i] == a and chunk[i + 1] == b:
                new_chunk.append(new_id)
                i += 2
            else:
                new_chunk.append(chunk[i])
                i += 1
        new_chunks.append(tuple(new_chunk))
    return new_chunks


def train_bpe(text, num_merges):
    chunks = get_chunks(text)

    # Vocabulary starts as all 256 possible bytes
    vocab = {i: bytes([i]) for i in range(256)}
    merges = {}  # (pair) -> new_id

    for i in range(num_merges):
        counts = get_pair_counts(chunks)
        if not counts:
            break

        best_pair = max(counts, key=counts.get)
        new_id = 256 + i

        chunks = merge_chunks(chunks, best_pair, new_id)
        merges[best_pair] = new_id
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]

        print(f"Merge {i + 1}: {best_pair} -> {new_id}  ({vocab[new_id]})")

    return merges, vocab


def encode(text, merges):
    chunks = get_chunks(text)

    for pair, new_id in merges.items():
        chunks = merge_chunks(chunks, pair, new_id)

    return [token for chunk in chunks for token in chunk]


def decode(ids, vocab):
    return b"".join(vocab[i] for i in ids).decode("utf-8")

In [3]:
merges, vocab = train_bpe(text, num_merges=100)

Merge 1: (32, 32) -> 256  (b'  ')
Merge 2: (32, 116) -> 257  (b' t')
Merge 3: (257, 104) -> 258  (b' th')
Merge 4: (32, 98) -> 259  (b' b')
Merge 5: (256, 256) -> 260  (b'    ')
Merge 6: (105, 110) -> 261  (b'in')
Merge 7: (114, 101) -> 262  (b're')
Merge 8: (44, 10) -> 263  (b',\n')
Merge 9: (32, 119) -> 264  (b' w')
Merge 10: (32, 97) -> 265  (b' a')
Merge 11: (32, 115) -> 266  (b' s')
Merge 12: (108, 100) -> 267  (b'ld')
Merge 13: (32, 111) -> 268  (b' o')
Merge 14: (101, 115) -> 269  (b'es')
Merge 15: (32, 109) -> 270  (b' m')
Merge 16: (32, 102) -> 271  (b' f')
Merge 17: (259, 101) -> 272  (b' be')
Merge 18: (39, 115) -> 273  (b"'s")
Merge 19: (111, 117) -> 274  (b'ou')
Merge 20: (110, 100) -> 275  (b'nd')
Merge 21: (104, 101) -> 276  (b'he')
Merge 22: (258, 121) -> 277  (b' thy')
Merge 23: (32, 99) -> 278  (b' c')
Merge 24: (32, 100) -> 279  (b' d')
Merge 25: (258, 101) -> 280  (b' the')
Merge 26: (260, 260) -> 281  (b'        ')
Merge 27: (115, 101) -> 282  (b'se')
Merge 28: (32

In [4]:
ids = encode(text, merges)

text_dec = decode(ids, vocab)

text_dec == text

True

In [16]:
import rustbpe
import tiktoken


tokenizer = rustbpe.Tokenizer()
tokenizer.train_from_iterator(text, vocab_size=8192, pattern=SPLIT_PATTERN)

encoder = tiktoken.Encoding(
    name="rustbpe",
    pat_str=tokenizer.get_pattern(),
    mergeable_ranks={bytes(k): v for k, v in tokenizer.get_mergeable_ranks()},
    special_tokens={},
)

ids = encoder.encode("Hello world")
text = encoder.decode(ids)